In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import *

In [0]:
customers_df = spark.read.table("bankaml.silver.customers").filter(col("is_current") == "Y")
accounts_df = spark.read.table("bankaml.silver.accounts").filter(col("is_current") == "Y")
transactions_df = spark.read.table("bankaml.silver.transactions")

In [0]:
## Accounts count for each customer
cust_accounts_df = (
    customers_df.alias("c").join(
        accounts_df.alias("a"),
        col("c.customer_id") == col("a.customer_id"),
        "left"
    )
    .select(
        col("c.customer_id").alias("customer_id"),
        col("c.risk_rating").alias("stated_risk_rating"),
        col("a.account_id").alias("account_id")
    )
)

## total transactions count and amount
txns_summary_per_cust = (
    accounts_df.alias("a").join(
        transactions_df.alias("t"),
        col("a.account_id") == col("t.account_id"),
        "inner"
    )
    .groupBy(col("a.customer_id"))
    .agg(
        countDistinct("a.account_id").cast(IntegerType()).alias("customer_account_count"),
        sum("t.amount_usd").cast(DecimalType(18,2)).alias("customer_total_txn_volume_usd"),
        countDistinct("t.txn_id").cast(IntegerType()).alias("customer_txn_count")
    ).select(
        col("a.customer_id").alias("customer_id"),
        col("customer_account_count"),
        col("customer_total_txn_volume_usd"),
        col("customer_txn_count")
    )
)

## Flags count for each customer
structuring_flags_df = spark.read.table("bankaml.gold.structuring_flags")
rapid_inout_flags_df = spark.read.table("bankaml.gold.rapid_inout_flags")

structuring_flags_count_df = (
    customers_df.alias("c").join(
        structuring_flags_df.alias("sf"), 
        col("c.customer_id") == col("sf.customer_id"),
        "inner"
    )
    .groupBy("c.customer_id")
    .agg(
        countDistinct("sf.flag_id").alias("customer_flag_count")
    ).select(
        col("c.customer_id").alias("customer_id"),
        col("customer_flag_count")
    )
)

rapid_inout_flags_count_df = (
    customers_df.alias("c").join(
        rapid_inout_flags_df.alias("rf"),
        col("c.customer_id") == col("rf.customer_id"),
        "inner"
    ).groupBy("c.customer_id")
    .agg(
        countDistinct("rf.flag_id").alias("customer_flag_count")
    ).select(
        col("c.customer_id").alias("customer_id"),
        col("customer_flag_count")
    )
)

flag_count_per_cust = (
    structuring_flags_count_df.unionByName(rapid_inout_flags_count_df)
    .groupBy(col("customer_id"))
    .agg((sum("customer_flag_count").alias("customer_flag_count")).cast(IntegerType()))
    .withColumn(
        "behavioral_risk_score", 
            when(col("customer_flag_count")>=3, lit("high"))
            .when(col("customer_flag_count")>1, lit("medium"))
            .otherwise(lit("low"))
    )
    .withColumn(
        "priority_review_flag",
        when(col("behavioral_risk_score") == "high", lit("Y"))
        .otherwise(lit("N"))
    )
)

risk_summary_df = (
    cust_accounts_df.alias("ca").join(
        txns_summary_per_cust.alias("t"),
        col("ca.customer_id") == col("t.customer_id"),
        "left"
    )
    .join(
        flag_count_per_cust.alias("f"),
        col("ca.customer_id") == col("f.customer_id"),
        "left"
    )
    .select(
        col("ca.customer_id").alias("customer_id"),
        col("ca.stated_risk_rating").alias("stated_risk_rating"),
        col("t.customer_account_count").alias("customer_account_count"),
        col("t.customer_total_txn_volume_usd").alias("customer_total_txn_volume_usd"),
        col("t.customer_txn_count").alias("customer_txn_count"),
        col("f.customer_flag_count").alias("customer_flag_count"),
        col("f.behavioral_risk_score").alias("behavioral_risk_score"),
        col("f.priority_review_flag").alias("priority_review_flag")
    )
)

In [0]:
%sql
create table if not exists bankaml.gold.customer_risk_summary
(
    customer_id string,
    stated_risk_rating string,
    customer_account_count int,
    customer_total_txn_volume_usd decimal(18,2),
    customer_txn_count int,
    customer_flag_count int,
    behavioral_risk_score string,
    priority_review_flag string
)
using delta;

In [0]:
risk_summary_df.write.format("delta").mode("overwrite").saveAsTable("bankaml.gold.customer_risk_summary")